# HAKE-MER — export F1 par classe (test)

Ré-entraîne les campagnes **Step~0, M1, M1+M2** (DistilBERT, 3 graines) avec le code qui enregistre `test.per_label` dans chaque `metrics.json`, puis agrège `reference/artifacts/per_class_ladder_distilbert_seeds_42_123_456.json`.

**Durée indicative:** ~2 h GPU (9 entraînements, même protocole que ch.~4).

Télécharger le zip final + cette `.ipynb` exécutée vers `reference/training_records/step_per_class/colab/`.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_baseline_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}
!./run_m1_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}
!./run_m1_m2_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}

In [ ]:
!./run_per_class_from_metrics.sh --seeds 42,123,456

In [ ]:
import json
import zipfile
from pathlib import Path
from google.colab import files

artifact = Path("reference/artifacts/per_class_ladder_distilbert_seeds_42_123_456.json")
c = json.loads(artifact.read_text(encoding="utf-8"))
print("configurations:", len(c["configurations"]))
for cfg in c["configurations"]:
    print(cfg["configuration"], "macro", f"{cfg['test_f1_macro_mean']:.4f}", "seeds", cfg["seeds_evaluated"])

zip_path = Path("/content/per_class_ladder_distilbert_seeds_42_123_456.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(artifact, artifact.name)
    for p in sorted(Path("runs").glob("distilbert_base_uncased_seed*_*/metrics.json")):
        if any(x in p.parent.name for x in ("baseline_plm", "_m1", "_m1_m2")) and "m3" not in p.parent.name:
            zf.write(p, f"{p.parent.name}/{p.name}")
files.download(str(zip_path))